### TABLA DH

| i | θᵢ | dᵢ (mm) | aᵢ (mm) | αᵢ |
|---:|---|---:|---:|---:|
| 1 | θ₁ | 134.75 | 0 | +90° |
| 2 | θ₂ − 90° | 0 | -110 | 0° |
| 3 | θ₃ | 0 | -96 | 0° |
| 4 | θ₄ − 90° | 63.4 | 0 | +90° |
| 5 | θ₅ + 90° | 75.05 | 0 | -90° |
| 6 | θ₆ | 50 | 0 | 0° |

In [17]:
import numpy as np


# ==========================================
# MATRIZ DENAVIT-HARTENBERG
# ==========================================

def dh_matrix(theta, d, a, alpha):

    theta = np.deg2rad(theta)
    alpha = np.deg2rad(alpha)

    return np.array([
        [
            np.cos(theta),
            -np.sin(theta) * np.cos(alpha),
            np.sin(theta) * np.sin(alpha),
            a * np.cos(theta)
        ],
        [
            np.sin(theta),
            np.cos(theta) * np.cos(alpha),
            -np.cos(theta) * np.sin(alpha),
            a * np.sin(theta)
        ],
        [
            0,
            np.sin(alpha),
            np.cos(alpha),
            d
        ],
        [
            0,
            0,
            0,
            1
        ]
    ])


# ==========================================
# CINEMÁTICA DIRECTA
# ==========================================

def fk(theta):

    # Ángulos físicos entregados por get_angles()
    t1, t2, t3, t4, t5, t6 = theta

    # Offsets de la tabla DH
    q1 = t1
    q2 = t2 - 90
    q3 = t3
    q4 = t4 - 90
    q5 = t5 + 90
    q6 = t6

    # Tabla DH
    A1 = dh_matrix(q1, 134.75, 0,    90)
    A2 = dh_matrix(q2, 0,     -110,   0)
    A3 = dh_matrix(q3, 0,      -96,   0)
    A4 = dh_matrix(q4, 63.4,     0,  90)
    A5 = dh_matrix(q5, 75.05,    0, -90)
    A6 = dh_matrix(q6, 50,       0,   0)

    T06 = A1 @ A2 @ A3 @ A4 @ A5 @ A6

    return T06


# ==========================================
# 1. LEER ÁNGULOS REALES DEL JETCOBOT
# ==========================================

q = mc.get_angles()

print("\n================================")
print("ÁNGULOS REALES DEL JETCOBOT")
print("================================")

print(q)


# ==========================================
# 2. CALCULAR FK
# ==========================================

T = fk(q)

x_pred = T[0, 3]
y_pred = T[1, 3]
z_pred = T[2, 3]


print("\n================================")
print("MATRIZ T06")
print("================================")

print(np.round(T, 3))


print("\n================================")
print("COORDENADAS CALCULADAS POR FK")
print("================================")

print(f"X = {x_pred:.2f} mm")
print(f"Y = {y_pred:.2f} mm")
print(f"Z = {z_pred:.2f} mm")


# ==========================================
# 3. LEER COORDENADAS DEL JETCOBOT
# ==========================================

coords = mc.get_coords()

print("\n================================")
print("COORDENADAS DEL JETCOBOT")
print("================================")

print(coords)


if coords is not None and len(coords) >= 3:

    x_real = coords[0]
    y_real = coords[1]
    z_real = coords[2]


    print("\nPosición JetCobot:")

    print(f"X = {x_real:.2f} mm")
    print(f"Y = {y_real:.2f} mm")
    print(f"Z = {z_real:.2f} mm")


    # ==========================================
    # 4. CALCULAR ERROR POR EJE
    # ==========================================

    error_x = abs(x_real - x_pred)
    error_y = abs(y_real - y_pred)
    error_z = abs(z_real - z_pred)


    # ==========================================
    # 5. ERROR CARTESIANO TOTAL
    # ==========================================

    error_total = np.sqrt(
        (x_real - x_pred) ** 2 +
        (y_real - y_pred) ** 2 +
        (z_real - z_pred) ** 2
    )


    # ==========================================
    # 6. MOSTRAR COMPARACIÓN
    # ==========================================

    print("\n================================")
    print("COMPARACIÓN FK VS JETCOBOT")
    print("================================")

    print(
        f"FK:       "
        f"({x_pred:.2f}, "
        f"{y_pred:.2f}, "
        f"{z_pred:.2f}) mm"
    )

    print(
        f"JetCobot: "
        f"({x_real:.2f}, "
        f"{y_real:.2f}, "
        f"{z_real:.2f}) mm"
    )




ÁNGULOS REALES DEL JETCOBOT
[1.84, 0.7, -0.61, -0.96, 0.61, -44.38]

MATRIZ T06
[[ 4.10000e-02  1.90000e-02  9.99000e-01  5.16290e+01]
 [-7.14000e-01 -6.99000e-01  4.30000e-02 -6.12420e+01]
 [ 6.99000e-01 -7.15000e-01 -1.50000e-02  4.15024e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]

COORDENADAS CALCULADAS POR FK
X = 51.63 mm
Y = -61.24 mm
Z = 415.02 mm

COORDENADAS DEL JETCOBOT
[50.2, -62.5, 410.0, -91.22, -44.36, -86.67]

Posición JetCobot:
X = 50.20 mm
Y = -62.50 mm
Z = 410.00 mm

COMPARACIÓN FK VS JETCOBOT
FK:       (51.63, -61.24, 415.02) mm
JetCobot: (50.20, -62.50, 410.00) mm


In [8]:
print(mc.get_coords())

[49.9, -62.5, 410.1, -91.13, -45.6, -86.73]


In [20]:
import time
import numpy as np

poses = [
    [0, 0, 0, 0, 0, 0],       # HOME
    [0, 30, -30, 0, 0, 0],
    [30, 20, -40, 20, 0, 0]
]

VELOCIDAD = 5

for i, q in enumerate(poses, start=1):

    print("\n================================")
    print(f"POSE {i}")
    print("================================")

    # ==========================================
    # 1. PREDICCIÓN ANTES DE MOVER EL ROBOT
    # ==========================================

    T = fk(q)

    x_pred = T[0, 3]
    y_pred = T[1, 3]
    z_pred = T[2, 3]

    print("\nÁngulos enviados:")
    print(q)

    print("\nCOORDENADAS PREDICHAS POR FK:")
    print(f"X = {x_pred:.2f} mm")
    print(f"Y = {y_pred:.2f} mm")
    print(f"Z = {z_pred:.2f} mm")

    # ==========================================
    # 2. MOVER JETCOBOT
    # ==========================================

    print("\nMoviendo JetCobot...")

    mc.send_angles(q, VELOCIDAD)

    time.sleep(8)

    # ==========================================
    # 3. LEER ÁNGULOS REALES
    # ==========================================

    q_real = mc.get_angles()

    print("\nÁngulos reales ejecutados:")
    print(q_real)

    # ==========================================
    # 4. LEER COORDENADAS DEL JETCOBOT
    # ==========================================

    coords = mc.get_coords()

    print("\nCoordenadas completas del JetCobot:")
    print(coords)

    if coords is None or len(coords) < 3:
        print("No se pudieron obtener las coordenadas.")
        continue

    x_real = coords[0]
    y_real = coords[1]
    z_real = coords[2]

    print("\nCOORDENADAS DEL JETCOBOT:")
    print(f"X = {x_real:.2f} mm")
    print(f"Y = {y_real:.2f} mm")
    print(f"Z = {z_real:.2f} mm")

    # ==========================================
    # 5. ERROR CARTESIANO
    # ==========================================

    error_x = abs(x_real - x_pred)
    error_y = abs(y_real - y_pred)
    error_z = abs(z_real - z_pred)

    error_total = np.sqrt(
        (x_real - x_pred) ** 2 +
        (y_real - y_pred) ** 2 +
        (z_real - z_pred) ** 2
    )

    # ==========================================
    # 6. RESULTADOS
    # ==========================================

    print("\n------------- COMPARACIÓN -------------")

    print(
        f"FK:       "
        f"({x_pred:.2f}, "
        f"{y_pred:.2f}, "
        f"{z_pred:.2f}) mm"
    )

    print(
        f"JetCobot: "
        f"({x_real:.2f}, "
        f"{y_real:.2f}, "
        f"{z_real:.2f}) mm"
    )


    print("================================")


POSE 1

Ángulos enviados:
[0, 0, 0, 0, 0, 0]

COORDENADAS PREDICHAS POR FK:
X = 50.00 mm
Y = -63.40 mm
Z = 415.80 mm

Moviendo JetCobot...

Ángulos reales ejecutados:
[0.96, 0.26, -0.61, -1.05, 0.7, -0.7]

Coordenadas completas del JetCobot:
[51.5, -63.1, 409.5, -91.4, -0.68, -88.31]

COORDENADAS DEL JETCOBOT:
X = 51.50 mm
Y = -63.10 mm
Z = 409.50 mm

------------- COMPARACIÓN -------------
FK:       (50.00, -63.40, 415.80) mm
JetCobot: (51.50, -63.10, 409.50) mm

POSE 2

Ángulos enviados:
[0, 30, -30, 0, 0, 0]

COORDENADAS PREDICHAS POR FK:
X = -5.00 mm
Y = -63.40 mm
Z = 401.06 mm

Moviendo JetCobot...

Ángulos reales ejecutados:
[0.79, 29.61, -29.79, -1.14, 0.61, -0.61]

Coordenadas completas del JetCobot:
[-3.1, -64.1, 395.2, -91.31, -0.6, -88.57]

COORDENADAS DEL JETCOBOT:
X = -3.10 mm
Y = -64.10 mm
Z = 395.20 mm

------------- COMPARACIÓN -------------
FK:       (-5.00, -63.40, 401.06) mm
JetCobot: (-3.10, -64.10, 395.20) mm

POSE 3

Ángulos enviados:
[30, 20, -40, 20, 0, 0]

COO